## Step 2: Peeking Inside a Star's Diary (FITS Exploration & Light Curve Extraction)

Now that we have our list of files, it's time to actually open one and see what a Kepler TPF is hiding. Spoiler: lots of `NaN`s and a very shy star. This notebook is essentially the *exploratory* / prototyping stage — figuring out how to go from "raw pixel blob" to "clean brightness-over-time signal" before we scale it up in Notebook 3.

**Background context:** Kepler doesn't give you a single brightness number per timestamp — it gives you a tiny image (a "postage stamp") of the star and its surroundings at every cadence. Somewhere in that image is the actual star, and somewhere else is empty sky/noise. The "aperture mask" tells you which pixels belong to the star.

**The Algorithm:**
1. **Load the roster** from `names.fitz.gz.csv` created in Notebook 1 — teamwork makes the dream work. `df.head(20)` is used to sanity-check that the file loaded correctly.
2. **Open a `.fits.gz` file** with `astropy.io.fits.open()` and grab the pixel-level `FLUX` data from HDU index `[1]` (shape: *time × 10 × 12* — basically a tiny flickering movie of a star, one frame per cadence).
3. **Resize each frame** from 10×12 down to **8×7** using `skimage.transform.resize(..., mode='reflect', preserve_range=True)`. This is done per-frame in a loop and re-stacked with `np.array()`. Reducing resolution here trades a bit of spatial detail for a smaller, more uniform image size — useful later when feeding into a CNN that expects fixed input dimensions.
4. **Convert to a Pandas DataFrame** via `astropy.table.Table(data).to_pandas()` — turning telescope-native FITS records (`TIME`, `FLUX`, `FLUX_ERR`, `QUALITY`, `POS_CORR1/2`, etc.) into something a spreadsheet enjoyer can love and slice with familiar Pandas syntax.
5. **Grab the aperture mask** from HDU index `[2]` (`hdulist[2].data`) — this is a small array of 0s and 1s (or similar flags) marking which pixels in the frame actually belong to the target star versus background sky.
6. **Apply the aperture mask**: for every flux frame, reshape it to match the aperture's shape, multiply element-wise by the mask, and `np.nansum()` the result → this gives you the star's total **brightness** per timestamp, ignoring background pixels and `NaN`s. Think of it as "add up only the pixels that are actually the star, not the neighboring void."
7. **Filter by `QUALITY == 0`** — Kepler flags cadences affected by cosmic ray hits, spacecraft attitude adjustments, safe-mode events, etc. with non-zero quality codes. Here, `df_c = df[df["QUALITY"] == 0].copy()` politely shows the junk cadences the door (in this dataset, that dropped it from 476 → 442 rows).
8. **Normalize brightness**: z-score it — `(brightness - mean) / std` — so the light curve isn't at the mercy of arbitrary flux units (e-/s) and different stars/files become comparable on the same scale.
9. **Detrend with a rolling median** (`window=400, center=True, min_periods=10`), then `bfill()/ffill()` the edges where the window can't fully form. This removes slow, boring stellar drift (e.g. gradual dimming from instrument thermal effects) so that a sudden *dip* (possible planet transit!) actually stands out instead of getting lost in the star's long-term mood swings.
10. **Plot everything** in a two-panel figure: raw brightness over time, and a second panel with normalized flux + trend overlay + the final detrended residual — because if it's not visualized, did it even happen?

**Output:** No files saved here — this is a scratchpad/prototyping notebook. The real payoff is the *technique* (resize → mask → quality-filter → normalize → detrend) that gets reused and scaled in Notebook 3.

*In short: this notebook turns a blurry space photo album into a clean, scientific "did something eclipse this star?" chart.*

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('names.fitz.gz.csv')

In [ ]:
df.head(20)

In [ ]:
from skimage.transform import resize
import numpy as np
from astropy.io import fits
from astropy.table import Table

hdulist = fits.open("kplr010874614-2009131105131_lpd-targ.fits.gz")
data = hdulist[1].data
flux = data['FLUX']  # Shape: (time, 10, 12)

# Resize all images to 8x7
resized_flux = []
for img in flux:
    img_resized = resize(img, (8, 7), mode='reflect', preserve_range=True)
    resized_flux.append(img_resized)

resized_flux = np.array(resized_flux)
print(f"Original shape: {flux.shape}")
print(f"Resized shape: {resized_flux.shape}")  # Should be (time, 8, 7)

In [ ]:
type(data)

In [ ]:
table = Table(data)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = table.to_pandas()

In [ ]:
df.head(1)

In [ ]:
df.columns

In [ ]:
aperture = hdulist[2].data

In [ ]:
brightness = []
for flux_img in df["FLUX"]:
    flux_2d = np.array(flux_img)
    if flux_2d.shape != aperture.shape:
        flux_2d = flux_2d.reshape(aperture.shape)
    masked_flux = flux_2d * aperture
    brightness.append(np.nansum(masked_flux))

In [ ]:
df["brightness"] = brightness

In [ ]:
df.QUALITY.unique()

In [ ]:
df_c = df[df["QUALITY"] == 0].copy()

In [ ]:
df.shape

In [ ]:
df_c.shape

In [ ]:
df_c["brightness_norm"] = (df_c["brightness"] - df_c["brightness"].mean()) / df_c["brightness"].std()

In [ ]:
window = 400
trend = df_c["brightness_norm"].rolling(window, center=True, min_periods=10).median()

In [ ]:
trend = trend.bfill().ffill()

In [ ]:
df_c["detrended"] = df_c["brightness_norm"] - trend

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df_c['TIME'], df_c['brightness'], '.', markersize=2)
plt.xlabel("Time")
plt.ylabel("Flux (Brightness)")
plt.title("Kepler Light Curve")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df_c['TIME'], df_c['brightness_norm'], '.', markersize=1, alpha=0.5, color='blue')
axes[0].plot(df_c['TIME'], trend, '-', linewidth=2, color='red', alpha=0.7)
axes[0].set_ylabel("Normalized Flux")
axes[0].set_title("Normalized Light Curve with Trend")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_c['TIME'], df_c['detrended'], '.', markersize=1, alpha=0.5, color='green')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel("Time (BJD - 2454833)")
axes[1].set_ylabel("Detrended Flux")
axes[1].set_title("Detrended Light Curve")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

hdulist.close()